# Lab 8: Synthetic InSAR — Seeing Faults from Space

> **Colab note:** This notebook is designed to run on **Google Colab**. [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amtseismo/EPS166/blob/main/notebooks/08_insar_lab.ipynb)

## Introduction

InSAR interferograms are one of the most information-rich geodetic observables we have — a single image can contain millions of displacement measurements across an entire fault system. But reading an interferogram takes practice. The pattern of interference fringes encodes fault geometry, slip magnitude, and depth in ways that are not always intuitive.

In this lab you will build a forward model that computes synthetic InSAR interferograms from fault parameters. Using the `cutde` elastic dislocation library (Meade 2007) — the same physics underlying Okada (1985) — you will systematically explore how strike, dip, rake, depth, and slip magnitude change the interferogram pattern. By the end you should be able to look at a real interferogram and make an informed guess about the fault geometry that produced it.

This lab is inspired by the Visible Earthquakes tool and the GETSI Imaging Active Tectonics module (Unit 4).

## Learning objectives

By the end, you will be able to:

- build a synthetic InSAR interferogram from fault parameters using elastic dislocation theory
- interpret interference fringes in terms of line-of-sight displacement
- explain how satellite viewing geometry (ascending vs. descending, incidence angle) affects the observed pattern
- recognize the characteristic interferogram patterns of strike-slip, normal, and thrust faults
- predict how depth, slip magnitude, and fault size change the fringe pattern
- connect synthetic patterns to real Ridgecrest InSAR observations

## Notebook outline
- [Part I: The forward model](#part-i-the-forward-model)
- [Part II: A guided example](#part-ii-a-guided-example)
- [Part III: Exploring fault geometry](#part-iii-exploring-fault-geometry)
- [Part IV: Application](#part-iv-application)
- [Synthesis](#synthesis)
- [Summary](#summary)


## Setup

Install `cutde` and import the packages used throughout this lab.


In [ ]:
%pip install -q cutde

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cutde.halfspace as hs
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, fixed

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.titlesize': 11,
})

# Sentinel-1 C-band wavelength
WAVELENGTH = 0.056   # meters
FRINGE_M   = WAVELENGTH / 2  # one fringe = half-wavelength of LOS motion

print(f"C-band wavelength: {WAVELENGTH*100:.1f} cm")
print(f"One fringe = {FRINGE_M*100:.1f} cm of LOS displacement")


## Part I: The forward model

### Building the forward model

We use `cutde` to compute elastic displacements from a rectangular fault. `cutde` represents the fault as two triangles and uses the Meade (2007) triangular dislocation element formulation — equivalent to Okada (1985) for rectangular faults, but more flexible.

**Coordinate system:** x = East, y = North, z = Up (positive). Depth is negative z.

**Fault parameters:**
- **Strike** (°): direction of the fault trace, measured clockwise from North
- **Dip** (°): angle of the fault surface below horizontal (0° = horizontal, 90° = vertical)
- **Rake** (°): direction of slip on the fault plane (0° = left-lateral, 90° = reverse, 180° = right-lateral, -90° = normal)
- **Length** (km): along-strike extent of the fault
- **Width** (km): down-dip extent of the fault
- **Depth to top** (km): depth to the shallowest edge
- **Slip** (m): average slip magnitude


In [ ]:
def build_fault_triangles(strike_deg, dip_deg, rake_deg,
                           length_km, width_km, depth_top_km, slip_m=1.0):
    """
    Build a rectangular fault as two triangles for cutde.
    All distances in METERS internally (cutde uses consistent units).
    The fault is centered at the surface origin (x=0, y=0).

    Parameters
    ----------
    strike_deg : float  Fault strike, degrees clockwise from North
    dip_deg    : float  Fault dip, degrees below horizontal
    rake_deg   : float  Slip rake on fault plane
    length_km  : float  Along-strike length in km
    width_km   : float  Down-dip width in km
    depth_top_km : float  Depth to the top edge in km
    slip_m     : float  Slip magnitude in meters

    Returns
    -------
    tris  : (2, 3, 3) array  Triangle vertices in meters
    slips : (2, 3) array     Strike-slip, dip-slip, tensile slip per triangle
    """
    s = np.radians(strike_deg)
    d = np.radians(dip_deg)
    r = np.radians(rake_deg)

    # Unit vectors
    along_strike = np.array([np.sin(s),  np.cos(s), 0.0])
    down_dip     = np.array([np.cos(d)*np.cos(s + np.pi/2),
                             -np.cos(d)*np.sin(s + np.pi/2),
                             -np.sin(d)])

    # Fault corners in meters (fault centered at origin)
    L = length_km * 1e3
    W = width_km  * 1e3
    D = depth_top_km * 1e3
    center_top = np.array([0.0, 0.0, -D])

    P0 = center_top - L/2 * along_strike
    P1 = center_top + L/2 * along_strike
    P2 = P1 + W * down_dip
    P3 = P0 + W * down_dip

    tris = np.array([[P0, P1, P2], [P0, P2, P3]], dtype=float)

    # Slip components (cutde convention: positive strike-slip = left-lateral)
    slip_strike  =  slip_m * np.cos(r)
    slip_dip     = -slip_m * np.sin(r)  # positive = reverse
    slips = np.tile([slip_strike, slip_dip, 0.0], (2, 1))

    return tris, slips


def los_unit_vector(inc_deg=38.0, head_deg=-12.0):
    """
    Compute the line-of-sight unit vector pointing FROM ground TO satellite.

    Parameters
    ----------
    inc_deg  : float  Incidence angle from vertical (degrees)
    head_deg : float  Satellite heading (degrees from North, clockwise)
                      Ascending track ≈ -12° (northward flight, looking west)
                      Descending track ≈ 192° (southward flight, looking east)

    Returns
    -------
    los : (3,) array  [East, North, Up] components of unit LOS vector
    """
    inc  = np.radians(inc_deg)
    head = np.radians(head_deg)
    los = np.array([
        -np.sin(inc) * np.sin(head),   # East
        -np.sin(inc) * np.cos(head),   # North
         np.cos(inc)                   # Up
    ])
    return los


def compute_interferogram(strike_deg=0, dip_deg=90, rake_deg=0,
                           length_km=30, width_km=15, depth_top_km=3,
                           slip_m=2.0, grid_km=80, nx=300,
                           inc_deg=38.0, head_deg=-12.0,
                           wavelength=WAVELENGTH):
    """
    Compute a synthetic InSAR interferogram.

    Returns
    -------
    X_km, Y_km : (nx, nx) grids of coordinates in km
    los_cm     : LOS displacement in cm (positive = toward satellite)
    wrapped    : Wrapped phase in radians (-pi to pi)
    """
    los = los_unit_vector(inc_deg, head_deg)

    # Observation grid in meters
    x = np.linspace(-grid_km * 1e3, grid_km * 1e3, nx)
    X, Y = np.meshgrid(x, x)
    obs  = np.column_stack([X.ravel(), Y.ravel(), np.zeros(X.size)])

    tris, slips = build_fault_triangles(
        strike_deg, dip_deg, rake_deg,
        length_km, width_km, depth_top_km, slip_m
    )

    # Elastic displacement (meters)
    disp   = hs.disp_free(obs, tris, slips, nu=0.25)

    # LOS projection and wrapping
    los_m   = (disp @ los).reshape(X.shape)
    phase   = (4 * np.pi / wavelength) * los_m
    wrapped = np.angle(np.exp(1j * phase))

    return X/1e3, Y/1e3, los_m*100, wrapped  # km, km, cm, rad


print("Forward model functions defined.")
print(f"LOS vector (Sentinel-1 ascending): {los_unit_vector()}")  
print(f"  E={los_unit_vector()[0]:.3f}, N={los_unit_vector()[1]:.3f}, U={los_unit_vector()[2]:.3f}")
print("\n→ InSAR is most sensitive to vertical (U) and east-west (E) displacement.")
print("→ It has little sensitivity to north-south (N) displacement.")


### Understanding LOS geometry

Before computing interferograms, let's build intuition for what InSAR can and cannot see.

The **line-of-sight (LOS)** direction is the unit vector from the ground to the satellite. For Sentinel-1 in ascending mode (flying northward, looking right/west):

$$\hat{\mathbf{r}}_{LOS} = \begin{pmatrix} -\sin\theta_i \sin\alpha \\ -\sin\theta_i \cos\alpha \\ \cos\theta_i \end{pmatrix}$$

where $\theta_i$ is the incidence angle (~38°) and $\alpha$ is the heading (~-12° from North for ascending).

The observed LOS displacement is:
$$\Delta LOS = \mathbf{u} \cdot \hat{\mathbf{r}}_{LOS} = u_E \hat{r}_E + u_N \hat{r}_N + u_U \hat{r}_U$$

Positive LOS = ground moved **toward** the satellite (range decrease).


In [ ]:
# Visualize the LOS sensitivity
inc_deg  = 38.0
head_asc = -12.0    # ascending: satellite flying NNW, looking WSW
head_dsc = 192.0    # descending: satellite flying SSE, looking ESE

los_asc = los_unit_vector(inc_deg, head_asc)
los_dsc = los_unit_vector(inc_deg, head_dsc)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

components = ['East', 'North', 'Up']
colors     = ['#e41a1c', '#377eb8', '#4daf4a']

for ax, los, title in zip(axes, [los_asc, los_dsc],
                           ['Ascending (heading ≈ -12°)',
                            'Descending (heading ≈ 192°)']):
    bars = ax.bar(components, np.abs(los), color=colors, alpha=0.8, edgecolor='k')
    for bar, val in zip(bars, los):
        sign = '+' if val > 0 else ''
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{sign}{val:.3f}', ha='center', va='bottom', fontsize=10)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('|LOS sensitivity|')
    ax.set_title(f'Sentinel-1 {title}\n(incidence angle = {inc_deg}°)')
    ax.axhline(0.5, color='0.6', ls='--', lw=0.8)

plt.suptitle('LOS unit vector components — what InSAR sees', fontsize=12)
plt.tight_layout()
plt.show()

print("Key insight:")
print(f"  Ascending  — Up sensitivity: {los_asc[2]:.2f}, East: {los_asc[0]:.2f}, North: {los_asc[1]:.2f}")
print(f"  Descending — Up sensitivity: {los_dsc[2]:.2f}, East: {los_dsc[0]:.2f}, North: {los_dsc[1]:.2f}")
print()
print("→ Both tracks are most sensitive to vertical (Up) motion.")
print("→ East sensitivity changes sign between ascending and descending.")
print("→ North sensitivity is very small in both tracks.")


> **LOS geometry questions:**
> 1. A station moves 10 cm eastward, 0 cm northward, 0 cm vertically. What LOS displacement would Sentinel-1 ascending record? What about descending?
> 2. A station moves 0 cm eastward, 10 cm northward, 0 cm vertically. What LOS displacement would each track record?
> 3. A station moves 0 cm eastward, 0 cm northward, 10 cm upward. What LOS displacement would each track record?
> 4. Why is InSAR poorly suited to measuring north-south displacements? What orbital geometry change would fix this?


## Part II: A guided example

### A strike-slip fault

Let's start with a simple example — a vertical right-lateral strike-slip fault striking north-south, similar to the San Andreas or the North Anatolian Fault.

**Parameters:**
- Strike = 0° (north-south)
- Dip = 90° (vertical)
- Rake = 180° (right-lateral)
- Length = 40 km, Width = 15 km
- Depth to top = 0 km (surface rupture)
- Slip = 2 m


In [ ]:
def plot_interferogram(X, Y, los_cm, wrapped,
                        title='', fault_label='',
                        show_fault_trace=True, strike_deg=0,
                        length_km=40, ax_los=None, ax_ifg=None):
    """
    Plot LOS displacement and wrapped interferogram side by side.
    """
    standalone = ax_los is None
    if standalone:
        fig, (ax_los, ax_ifg) = plt.subplots(1, 2, figsize=(12, 5))

    # LOS displacement
    lim = np.percentile(np.abs(los_cm), 99)
    c1 = ax_los.contourf(X, Y, los_cm, levels=100,
                          cmap='RdBu_r', vmin=-lim, vmax=lim)
    if standalone:
        plt.colorbar(c1, ax=ax_los, label='LOS displacement (cm)')
    ax_los.set_xlabel('East (km)'); ax_los.set_ylabel('North (km)')
    ax_los.set_aspect('equal')
    ax_los.set_title(f'LOS displacement\n{fault_label}' if fault_label else 'LOS displacement')

    # Mark fault trace
    if show_fault_trace:
        s = np.radians(strike_deg)
        dx = np.sin(s) * length_km/2
        dy = np.cos(s) * length_km/2
        ax_los.plot([-dx, dx], [-dy, dy], 'k-', lw=2, label='Fault trace')
        ax_ifg_obj = ax_ifg  # reference for fault line below

    # Wrapped interferogram
    c2 = ax_ifg.contourf(X, Y, wrapped, levels=128,
                          cmap='hsv', vmin=-np.pi, vmax=np.pi)
    if standalone:
        cbar = plt.colorbar(c2, ax=ax_ifg, ticks=[-np.pi, 0, np.pi])
        cbar.set_ticklabels(['-π', '0', 'π'])
        cbar.set_label('Phase (rad)')
    ax_ifg.set_xlabel('East (km)'); ax_ifg.set_ylabel('North (km)')
    ax_ifg.set_aspect('equal')
    ax_ifg.set_title(f'Interferogram (each fringe = {FRINGE_M*100:.1f} cm LOS)\n{fault_label}'
                      if fault_label else f'Interferogram (each fringe = {FRINGE_M*100:.1f} cm)')

    if show_fault_trace:
        ax_ifg.plot([-dx, dx], [-dy, dy], 'k-', lw=2)

    if standalone:
        if title:
            fig.suptitle(title, fontsize=12, y=1.01)
        plt.tight_layout()
        plt.show()


# Guided example: right-lateral strike-slip
X, Y, los_cm, wrapped = compute_interferogram(
    strike_deg=0, dip_deg=90, rake_deg=180,
    length_km=40, width_km=15, depth_top_km=0,
    slip_m=2.0, grid_km=80, nx=300
)

plot_interferogram(X, Y, los_cm, wrapped,
                   title='Right-lateral strike-slip fault (strike=0°, dip=90°, rake=180°)',
                   fault_label='Strike=0°, Dip=90°, Rake=180°, Slip=2m, Depth=0km',
                   strike_deg=0, length_km=40)

# Count fringes
max_los = np.abs(los_cm).max()
n_fringes = max_los / (FRINGE_M * 100)
print(f"Maximum LOS displacement: {max_los:.1f} cm")
print(f"Number of fringes from center to edge: ~{n_fringes:.0f}")
print(f"Expected for 2m slip, surface rupture: consistent? {'Yes' if 5 < n_fringes < 80 else 'Check'}")


> **Interpret the strike-slip interferogram:**
> 1. Describe the overall pattern — how many lobes do you see? What is the maximum displacement on each side of the fault?
> 2. On the east side of the fault, is the ground moving toward or away from the satellite (ascending track)? How can you tell from the fringe colors?
> 3. Count the fringes on one side of the fault. Each fringe = 2.8 cm LOS. What is the total LOS displacement on that side?
> 4. The fringes are densest immediately adjacent to the fault. What does this mean in terms of the displacement gradient?
> 5. Notice the north-south asymmetry (the pattern is not symmetric about the x-axis). What causes this asymmetry given the LOS geometry we computed in Section 3?


## Part III: Exploring fault geometry

### Fault type — strike-slip, normal, and thrust

Now compare the three main fault types — strike-slip, normal, and thrust — for the same fault dimensions and slip.

All faults have the same north-south strike. The only change is the dip and rake.


In [ ]:
FAULT_TYPES = [
    dict(dip=90, rake=180,  label='Right-lateral\nstrike-slip\n(dip=90°, rake=180°)'),
    dict(dip=60, rake=-90,  label='Normal fault\n(dip=60°, rake=-90°)'),
    dict(dip=30, rake=90,   label='Thrust fault\n(dip=30°, rake=90°)'),
]

COMMON = dict(strike_deg=0, length_km=30, width_km=15,
              depth_top_km=3, slip_m=2.0, grid_km=70, nx=250)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for j, ft in enumerate(FAULT_TYPES):
    X, Y, los_cm, wrapped = compute_interferogram(
        dip_deg=ft['dip'], rake_deg=ft['rake'], **COMMON
    )
    lim = np.percentile(np.abs(los_cm), 99)

    # LOS
    c1 = axes[0, j].contourf(X, Y, los_cm, levels=80,
                              cmap='RdBu_r', vmin=-lim, vmax=lim)
    plt.colorbar(c1, ax=axes[0, j], label='cm')
    axes[0, j].set_title(f'{ft["label"]}', fontsize=10)
    axes[0, j].set_aspect('equal')
    axes[0, j].set_xlabel('East (km)')
    if j == 0: axes[0, j].set_ylabel('North (km) — LOS displacement')

    # Fault trace (project to surface)
    for ax in [axes[0,j], axes[1,j]]:
        ax.plot([-15, 15], [0, 0], 'k-', lw=2, label='Fault trace')

    # Interferogram
    c2 = axes[1, j].contourf(X, Y, wrapped, levels=128,
                              cmap='hsv', vmin=-np.pi, vmax=np.pi)
    axes[1, j].set_aspect('equal')
    axes[1, j].set_xlabel('East (km)')
    if j == 0: axes[1, j].set_ylabel('North (km) — Wrapped interferogram')

fig.suptitle('Three fault types — same strike (N-S), dimensions, and slip (2 m)\n'
             'Sentinel-1 ascending geometry (incidence 38°)', fontsize=12)
plt.tight_layout()
plt.show()


> **Compare fault types:**
> 1. Describe the key differences between the strike-slip, normal, and thrust interferograms. How many lobes does each have? Are they symmetric?
> 2. The normal fault pattern is **not** symmetric east-west. Why? Think about which side of the fault is the hanging wall and which is the footwall, and how each side moves.
> 3. The thrust fault produces fringes mostly on one side. Why? Which side moves toward the satellite, and why?
> 4. For the normal fault, the pattern looks like a bull's-eye (closed fringes). When would you see a sharp discontinuity across the fault instead? Change `depth_top_km` to 0 and re-run — what changes?
> 5. If you saw this normal fault interferogram in a paper without any labels, how would you determine whether it was a normal fault or a thrust fault? What additional information would help?


### Exploring fault parameters

Use the interactive sliders to explore how each parameter changes the interferogram pattern. **For each parameter, make a prediction first, then check it.**


In [ ]:
def interactive_interferogram(strike=0, dip=90, rake=180,
                               length_km=30, width_km=15,
                               depth_top_km=3, slip_m=2.0):
    X, Y, los_cm, wrapped = compute_interferogram(
        strike_deg=strike, dip_deg=dip, rake_deg=rake,
        length_km=length_km, width_km=width_km,
        depth_top_km=depth_top_km, slip_m=slip_m,
        grid_km=80, nx=250
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

    lim = max(np.percentile(np.abs(los_cm), 99), 1)
    c1  = axes[0].contourf(X, Y, los_cm, levels=80, cmap='RdBu_r', vmin=-lim, vmax=lim)
    plt.colorbar(c1, ax=axes[0], label='LOS displacement (cm)')
    axes[0].set_title('LOS displacement')
    axes[0].set_xlabel('East (km)'); axes[0].set_ylabel('North (km)')
    axes[0].set_aspect('equal')

    # Fault surface trace
    s  = np.radians(strike)
    dx = np.sin(s) * length_km / 2
    dy = np.cos(s) * length_km / 2
    for ax in axes:
        ax.plot([-dx, dx], [-dy, dy], 'k-', lw=2.5)

    c2 = axes[1].contourf(X, Y, wrapped, levels=128, cmap='hsv',
                           vmin=-np.pi, vmax=np.pi)
    plt.colorbar(c2, ax=axes[1], label='Phase (rad)')
    axes[1].set_title(f'Wrapped interferogram — each fringe = {FRINGE_M*100:.1f} cm LOS')
    axes[1].set_xlabel('East (km)'); axes[1].set_ylabel('North (km)')
    axes[1].set_aspect('equal')

    n_fringes = np.abs(los_cm).max() / (FRINGE_M * 100)
    fig.suptitle(
        f'Strike={strike}°  Dip={dip}°  Rake={rake}°  '
        f'L={length_km}km  W={width_km}km  '
        f'Depth={depth_top_km}km  Slip={slip_m}m  '
        f'[~{n_fringes:.0f} fringes max]',
        fontsize=10
    )
    plt.tight_layout()
    plt.show()


interact(
    interactive_interferogram,
    strike      = IntSlider(min=0,    max=360, step=5,   value=0,   description='Strike (°)',    style={'description_width':'initial'}),
    dip         = IntSlider(min=1,    max=90,  step=5,   value=90,  description='Dip (°)',        style={'description_width':'initial'}),
    rake        = IntSlider(min=-180, max=180, step=10,  value=180, description='Rake (°)',       style={'description_width':'initial'}),
    length_km   = FloatSlider(min=5,  max=200, step=5,   value=30,  description='Length (km)',    style={'description_width':'initial'}),
    width_km    = FloatSlider(min=5,  max=80,  step=5,   value=15,  description='Width (km)',     style={'description_width':'initial'}),
    depth_top_km= FloatSlider(min=0,  max=30,  step=1,   value=3,   description='Depth top (km)',style={'description_width':'initial'}),
    slip_m      = FloatSlider(min=0.1,max=10,  step=0.1, value=2.0, description='Slip (m)',       style={'description_width':'initial'}),
);


> **Parameter exploration — make a prediction before each experiment:**
>
> **Strike:** Set strike to 0°, then rotate to 45°, 90°, 135°.
> 1. How does the interferogram pattern rotate relative to the fault trace? Does it rotate with the fault or against it?
>
> **Depth:** Start with depth = 0 km (surface rupture), then increase to 5, 10, 20 km.
> 2. What happens to the fringe density near the fault as you increase depth? What happens to the spatial extent of the deformation?
> 3. At what depth does the sharp surface discontinuity disappear and the pattern become smooth and closed?
>
> **Slip magnitude:** Keep geometry fixed (dip=90, rake=180, depth=5 km), vary slip from 0.5 to 5 m.
> 4. How does the number of fringes change with slip? Is the relationship linear? Why?
> 5. What is the minimum slip you could detect if you needed at least 1 fringe of signal?
>
> **Fault size:** Keep slip fixed at 2 m, vary length from 10 to 100 km.
> 6. How does the spatial extent of the deformation pattern scale with fault length?
>
> **Rake:** Set dip=45°, explore rake from -180° to 180°.
> 7. At what rake values do you see the most symmetric vs. most asymmetric patterns?
> 8. Find the rake that produces a bull's-eye pattern (concentric closed fringes). What fault type does this correspond to?


### Ascending vs. descending tracks

InSAR satellites acquire data on both ascending (southward-to-northward) and descending (northward-to-southward) orbital passes. Because the satellite looks from different directions, the same ground motion appears differently in each track.


In [ ]:
# Compare ascending and descending for a strike-slip fault
# Sentinel-1 typical geometries
ASC  = dict(inc_deg=38, head_deg=-12,  label='Ascending\n(looking WSW)')
DESC = dict(inc_deg=38, head_deg=192,  label='Descending\n(looking ESE)')

# Try different fault types
FAULTS = [
    dict(strike_deg=0, dip_deg=90, rake_deg=180, name='Right-lateral N-S strike-slip'),
    dict(strike_deg=0, dip_deg=60, rake_deg=-90, name='Normal fault (N-S strike, 60° dip W)'),
    dict(strike_deg=90, dip_deg=90, rake_deg=0,  name='Left-lateral E-W strike-slip'),
]

fig, axes = plt.subplots(len(FAULTS), 2, figsize=(12, 5*len(FAULTS)))

for i, fault in enumerate(FAULTS):
    for j, geom in enumerate([ASC, DESC]):
        X, Y, los_cm, wrapped = compute_interferogram(
            strike_deg=fault['strike_deg'],
            dip_deg=fault['dip_deg'],
            rake_deg=fault['rake_deg'],
            length_km=30, width_km=15, depth_top_km=3, slip_m=2.0,
            grid_km=70, nx=250,
            inc_deg=geom['inc_deg'], head_deg=geom['head_deg']
        )
        lim = max(np.percentile(np.abs(los_cm), 99), 1)
        c = axes[i, j].contourf(X, Y, los_cm, levels=80,
                                 cmap='RdBu_r', vmin=-lim, vmax=lim)
        plt.colorbar(c, ax=axes[i, j], label='LOS (cm)')

        # Fault trace
        s  = np.radians(fault['strike_deg'])
        dx = np.sin(s) * 15; dy = np.cos(s) * 15
        axes[i, j].plot([-dx, dx], [-dy, dy], 'k-', lw=2)
        axes[i, j].set_aspect('equal')
        axes[i, j].set_xlabel('East (km)')
        axes[i, j].set_ylabel('North (km)')
        axes[i, j].set_title(f"{fault['name']}\n{geom['label']}", fontsize=9)

plt.suptitle('LOS displacement: ascending vs. descending\n'
             'Note how the pattern changes with viewing geometry', fontsize=12)
plt.tight_layout()
plt.show()


> **Ascending vs. descending:**
> 1. For the right-lateral N-S strike-slip fault: the ascending track sees large signal on one side, the descending track sees a different pattern. Explain why in terms of the LOS vector components you computed in Section 3.
> 2. For the E-W left-lateral fault: why does the pattern look similar on both tracks? Which displacement component are both tracks sensitive to for this fault orientation?
> 3. Suppose you only had an ascending interferogram for a fault and needed to determine whether the motion was mostly horizontal or mostly vertical. How would you approach this? What would combining ascending and descending tell you?
> 4. In 2D decomposition using ascending + descending InSAR, we can recover the east and vertical components of displacement. Write down the two equations relating LOS_asc and LOS_dsc to (u_E, u_U), ignoring north motion. Could you solve for both unknowns?


## Part IV: Application

### Mystery interferograms

Below are four synthetic interferograms generated with unknown fault parameters. Work out the fault geometry from the interferogram pattern alone, then check your answer by entering parameters into the interactive tool in Part III.

**Approach:**
1. Count the fringes to estimate the LOS displacement magnitude
2. Identify the overall pattern shape (symmetric/asymmetric, lobes, bull's-eye)
3. Infer the likely fault type from the pattern
4. Estimate the strike from the orientation of the pattern
5. Estimate whether the fault is shallow (sharp discontinuity) or deep (smooth closed fringes)


In [ ]:
# Mystery faults — parameters hidden
# Students should not look at this cell until after making their predictions!
import hashlib

MYSTERIES = [
    dict(strike_deg=45,  dip_deg=90,  rake_deg=180, length_km=35, width_km=15,
         depth_top_km=0, slip_m=1.5),
    dict(strike_deg=0,   dip_deg=45,  rake_deg=-90, length_km=25, width_km=15,
         depth_top_km=5, slip_m=2.5),
    dict(strike_deg=90,  dip_deg=30,  rake_deg=90,  length_km=50, width_km=20,
         depth_top_km=3, slip_m=3.0),
    dict(strike_deg=315, dip_deg=90,  rake_deg=0,   length_km=40, width_km=15,
         depth_top_km=2, slip_m=2.0),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for j, m in enumerate(MYSTERIES):
    X, Y, los_cm, wrapped = compute_interferogram(
        grid_km=80, nx=250, **m
    )
    lim = max(np.percentile(np.abs(los_cm), 99), 1)

    axes[0, j].contourf(X, Y, los_cm, levels=80, cmap='RdBu_r', vmin=-lim, vmax=lim)
    axes[0, j].set_aspect('equal')
    axes[0, j].set_title(f'Mystery {j+1}\nLOS displacement', fontsize=10)
    axes[0, j].set_xlabel('East (km)')
    if j == 0: axes[0, j].set_ylabel('North (km)')

    axes[1, j].contourf(X, Y, wrapped, levels=128, cmap='hsv',
                        vmin=-np.pi, vmax=np.pi)
    axes[1, j].set_aspect('equal')
    axes[1, j].set_title(f'Mystery {j+1}\nInterferogram', fontsize=10)
    axes[1, j].set_xlabel('East (km)')
    if j == 0: axes[1, j].set_ylabel('North (km)')

plt.suptitle('Mystery interferograms — identify the fault type and parameters\n'
             '(Sentinel-1 ascending, incidence 38°, C-band)', fontsize=11)
plt.tight_layout()
plt.show()

print("Fill in your estimates before running the answer cell below!")
print()
print("Mystery 1: Strike=?  Dip=?  Rake=?  Depth_top=?  Slip=?")
print("Mystery 2: Strike=?  Dip=?  Rake=?  Depth_top=?  Slip=?")
print("Mystery 3: Strike=?  Dip=?  Rake=?  Depth_top=?  Slip=?")
print("Mystery 4: Strike=?  Dip=?  Rake=?  Depth_top=?  Slip=?")


In [ ]:
# ── Answers — run this cell only after making your estimates above ────────────
print("Mystery interferogram answers:")
print()
fault_types = ['Right-lateral NE strike-slip',
               'Normal fault (N-S strike, 45° dip)',
               'Thrust fault (E-W strike, 30° dip)',
               'Left-lateral NW strike-slip']
for j, (m, ft) in enumerate(zip(MYSTERIES, fault_types)):
    print(f"Mystery {j+1}: {ft}")
    print(f"  Strike={m['strike_deg']}°, Dip={m['dip_deg']}°, Rake={m['rake_deg']}°")
    print(f"  Length={m['length_km']} km, Width={m['width_km']} km")
    print(f"  Depth to top={m['depth_top_km']} km, Slip={m['slip_m']} m")
    print()


> **Mystery interferogram questions:**
> 1. For each mystery, describe the reasoning that led to your fault type estimate.
> 2. Which parameters were easiest to estimate from the interferogram? Which were hardest?
> 3. What is the fundamental ambiguity in determining fault geometry from a single InSAR interferogram that you cannot resolve without additional data?


### The Ridgecrest M7.1

Now apply your knowledge to the real 2019 Ridgecrest M7.1 earthquake. Based on what you learned in the seismology lab, the published fault geometry is:

| Parameter | Value |
|-----------|-------|
| Strike | ~322° (NW-striking) |
| Dip | ~85° (nearly vertical) |
| Rake | ~180° (right-lateral) |
| Length | ~50 km |
| Width | ~16 km |
| Depth to top | ~0 km (surface rupture) |
| Average slip | ~1.5 m |

Generate the synthetic interferogram and compare to the real Ridgecrest InSAR data from your course materials.


In [ ]:
# Ridgecrest M7.1 — published fault parameters
X, Y, los_cm, wrapped = compute_interferogram(
    strike_deg=322, dip_deg=85, rake_deg=180,
    length_km=50, width_km=16, depth_top_km=0,
    slip_m=1.5, grid_km=100, nx=300,
    inc_deg=38, head_deg=-12   # Sentinel-1 ascending
)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

lim = np.percentile(np.abs(los_cm), 99)
c1  = axes[0].contourf(X, Y, los_cm, levels=80, cmap='RdBu_r', vmin=-lim, vmax=lim)
plt.colorbar(c1, ax=axes[0], label='LOS displacement (cm)')
axes[0].set_title('LOS displacement (cm)\nRidgecrest M7.1 — ascending')
axes[0].set_xlabel('East (km)'); axes[0].set_ylabel('North (km)')
axes[0].set_aspect('equal')

# Fault trace
s = np.radians(322)
dx = np.sin(s)*25; dy = np.cos(s)*25
for ax in axes:
    ax.plot([-dx, dx], [-dy, dy], 'k-', lw=2.5, label='M7.1 fault trace')
    ax.legend(fontsize=8)

c2 = axes[1].contourf(X, Y, wrapped, levels=128, cmap='hsv', vmin=-np.pi, vmax=np.pi)
plt.colorbar(c2, ax=axes[1], label='Phase (rad)')
axes[1].set_title(f'Synthetic interferogram\nEach fringe = {FRINGE_M*100:.1f} cm LOS')
axes[1].set_xlabel('East (km)'); axes[1].set_ylabel('North (km)')
axes[1].set_aspect('equal')

n_fringes = lim / (FRINGE_M * 100)
plt.suptitle('Ridgecrest M7.1 synthetic InSAR\n'
             'Strike=322°, Dip=85°, Rake=180°, Length=50km, Width=16km, Slip=1.5m',
             fontsize=11)
plt.tight_layout()
plt.show()

print(f"Peak LOS displacement: {lim:.1f} cm")
print(f"Approximate fringe count from center to edge: ~{n_fringes:.0f}")
print(f"Compare to real Ridgecrest interferogram in your course materials.")


In [ ]:
# Now generate the DESCENDING track interferogram
X_d, Y_d, los_cm_d, wrapped_d = compute_interferogram(
    strike_deg=322, dip_deg=85, rake_deg=180,
    length_km=50, width_km=16, depth_top_km=0,
    slip_m=1.5, grid_km=100, nx=300,
    inc_deg=38, head_deg=192   # Sentinel-1 descending
)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
lim = np.percentile(np.abs(los_cm_d), 99)

c1 = axes[0].contourf(X_d, Y_d, los_cm_d, levels=80,
                       cmap='RdBu_r', vmin=-lim, vmax=lim)
plt.colorbar(c1, ax=axes[0], label='LOS displacement (cm)')
axes[0].set_title('LOS displacement — DESCENDING track')
axes[0].set_xlabel('East (km)'); axes[0].set_ylabel('North (km)')
axes[0].set_aspect('equal')

c2 = axes[1].contourf(X_d, Y_d, wrapped_d, levels=128,
                       cmap='hsv', vmin=-np.pi, vmax=np.pi)
plt.colorbar(c2, ax=axes[1], label='Phase (rad)')
axes[1].set_title('Synthetic interferogram — DESCENDING track')
axes[1].set_xlabel('East (km)'); axes[1].set_ylabel('North (km)')
axes[1].set_aspect('equal')

for ax in axes:
    ax.plot([-dx, dx], [-dy, dy], 'k-', lw=2.5)
    ax.set_aspect('equal')

plt.suptitle('Ridgecrest M7.1 — descending track\nCompare to ascending above', fontsize=11)
plt.tight_layout()
plt.show()


> **Ridgecrest questions:**
> 1. Compare your synthetic ascending interferogram to the real Ridgecrest interferogram from your course materials. What features match well? What features are missing or different in the synthetic?
> 2. Count the fringes in your synthetic interferogram from the fault to the edge of the image. How does this compare to the real interferogram?
> 3. The real Ridgecrest interferogram also shows signal from the M6.4 foreshock on a conjugate fault. Sketch where you would expect to see additional fringes if you added a second, NE-striking left-lateral fault to your model.
> 4. Compare the ascending and descending synthetic interferograms. On which side of the fault does the signal appear larger for ascending? For descending? Explain using the LOS geometry.
> 5. The uniform slip model predicts a smooth fringe pattern. The real interferogram shows more complexity near the fault. What physical factors does our simple model neglect?


## Synthesis

Write a short paragraph (3–5 sentences) answering each of the following.

> **1. Reading a real interferogram**  
> A colleague shows you an InSAR interferogram with the following features: symmetric bull's-eye pattern, 8 fringes from center to edge, no sharp discontinuity across the center, acquired on Sentinel-1 ascending geometry. Describe what you can infer about the fault — its type, approximate depth, and minimum displacement — and what you cannot determine from this single interferogram alone.

> **2. InSAR vs. GNSS**  
> In Lab 4 you measured coseismic GNSS offsets from the Ridgecrest M7.1 at a handful of stations. In this lab you computed synthetic InSAR covering the whole region at 300×300 pixels. What does each technique contribute that the other cannot? In what situation would you rely primarily on GNSS, and when on InSAR?

> **3. The north-south blind spot**  
> You are studying a large left-lateral fault oriented perfectly east-west (rake=0°, strike=90°). Your only available SAR data is Sentinel-1 ascending and descending. Explain why you might have difficulty detecting the earthquake signal, and suggest an alternative observation strategy.


## Summary

- **InSAR measures line-of-sight (LOS) displacement** — the component of ground motion toward or away from the satellite. Each fringe (one complete color cycle) represents $\lambda/2 = 2.8$ cm of LOS motion for C-band.

- **LOS sensitivity** depends on viewing geometry. Sentinel-1 is most sensitive to vertical and east-west displacement; north-south displacement is nearly invisible. Ascending and descending tracks sample different combinations of horizontal and vertical motion.

- **Fault type produces characteristic patterns:** strike-slip faults produce four-lobed antisymmetric patterns; normal and thrust faults produce asymmetric bull's-eye patterns whose asymmetry reflects the dip direction.

- **Depth controls fringe density and pattern shape.** Surface-rupturing faults produce a sharp discontinuity and very dense fringes near the fault; buried faults produce smooth closed fringes with no surface offset visible.

- **Slip scales the number of fringes** linearly — doubling the slip doubles the fringe count. Fault dimensions control the spatial extent of the deformation pattern.

- **The forward model we built here is the same one used in inversions.** In the inversion lectures, we will run this model millions of times to find the fault parameters that best fit a real interferogram — the opposite of what we did here.

### References

- Meade, B. J. (2007). Algorithms for the calculation of exact displacements, strains, and stresses for triangular dislocation elements in a uniform elastic half space. *Computers & Geosciences*, 33(8), 1064–1075. https://doi.org/10.1016/j.cageo.2006.12.003
- Okada, Y. (1985). Surface deformation due to shear and tensile faults in a half-space. *Bulletin of the Seismological Society of America*, 75(4), 1135–1154.
- Funning, G. (n.d.). *GEO 147: Active Tectonics and Remote Sensing* — InSAR lecture series.
- GETSI: Imaging Active Tectonics, Unit 4. https://serc.carleton.edu/getsi/teaching_materials/imaging_active_tectonics/unit4.html
- cutde library: https://github.com/cutde-org/cutde
